# 02_feature_engineering

## Imports

In [1]:
from recruit_restaurant_visitor_forecasting.config import (
    CALENDAR_DATE_COL,
    AIR_RESTAURANT_ID_COL,
    VISIT_DATE_COL,
    OPEN_DATE_COL,
    AIR_AREA_COL,
    CITY_COL,
    VISITORS_COL,
    AIR_GENRE_COL,
    GENRE_TE,
    AREA_TE,
    RESERVE_AIR,
    HPG_RESTAURANT_ID_COL,
    HPG_AREA_COL,
    RESERVE_HPG,
    RESERVE_AIR_NEIGHBORS,
    RESERVE_HPG_NEIGHBORS,
)

2025-11-29 21:39:19.170 | INFO     | recruit_restaurant_visitor_forecasting.config:<module>:13 - PROJ_ROOT path is: D:\mentoring_program


In [2]:
from recruit_restaurant_visitor_forecasting.dataset import (
    DataDir,
    read_csv,
    save_csv,
    prepare_datetime_columns,
    standardize_date
)

In [3]:
from recruit_restaurant_visitor_forecasting.features import (
    add_holiday_columns,
    add_golden_week_flg,
    add_opened_recently_flg,
    get_first_str_values,
    add_open_flg,
    add_days_since_last_record,
    add_time_based_target_encoding,
    add_sum_of_reserves,
    add_neighbors_reserves,
    add_total_reserves,
    add_total_neigh_reserves,
    add_basic_stats,
    add_neighbors_stats
)

In [4]:
air_visit_df = read_csv('air_visit.csv', DataDir.INTERIM)
air_reserve_df = read_csv('air_reserve.csv', DataDir.INTERIM)
hpg_reserve_df = read_csv('hpg_reserve.csv', DataDir.INTERIM)
sample_submission_df = read_csv('sample_submission.csv', DataDir.INTERIM)
date_info_df = read_csv('date_info.csv')
air_store_df = read_csv('air_store_info.csv')
hpg_store_df = read_csv('hpg_store_info.csv')
store_rel_df = read_csv('store_id_relation.csv')

In [5]:
prepare_datetime_columns(air_reserve_df)
prepare_datetime_columns(hpg_reserve_df)

standardize_date(date_info_df, CALENDAR_DATE_COL)
standardize_date(air_visit_df, VISIT_DATE_COL)
standardize_date(sample_submission_df, VISIT_DATE_COL)

## Features

### Air & hpg stores

#### City/region of area

For restaurants, it's best to separate the city and region (or, if necessary, just the city) without the district, since a relatively small number of restaurants belong to all three at once.

In [6]:
air_store_df[CITY_COL] = get_first_str_values(air_store_df[AIR_AREA_COL], 1)
hpg_store_df[CITY_COL] = get_first_str_values(hpg_store_df[HPG_AREA_COL], 1)

### Reserve dataframes

For further work, it is necessary to know the total number of reservations in the restaurant per day.

In [7]:
air_res_sum = add_sum_of_reserves(air_reserve_df, RESERVE_AIR, AIR_RESTAURANT_ID_COL)
air_res_sum = air_res_sum.merge(
    air_store_df[[AIR_RESTAURANT_ID_COL, CITY_COL]],
    on=AIR_RESTAURANT_ID_COL
)
air_res_sum = add_neighbors_reserves(air_res_sum, RESERVE_AIR, CITY_COL, RESERVE_AIR_NEIGHBORS)
air_res_sum.head()

,air_store_id,visit_date,air_reserves_sum,city,air_reserves_sum_neighbors
0,air_00a91d42b08b08d9,2016-10-31,2,Tōkyō-to,7.724138
1,air_00a91d42b08b08d9,2016-12-05,9,Tōkyō-to,8.095238
2,air_00a91d42b08b08d9,2016-12-14,18,Tōkyō-to,12.263158
3,air_00a91d42b08b08d9,2016-12-17,2,Tōkyō-to,15.536232
4,air_00a91d42b08b08d9,2016-12-20,4,Tōkyō-to,14.517857


In [8]:
air_res_sum[RESERVE_AIR_NEIGHBORS].isna().sum()

np.int64(515)

However, air_reserve dataframe still has a large number of gaps - I haven't filled them yet.

In [9]:
hpg_res_sum = add_sum_of_reserves(hpg_reserve_df, RESERVE_HPG, HPG_RESTAURANT_ID_COL)
hpg_res_sum = hpg_res_sum.merge(
    hpg_store_df[[HPG_RESTAURANT_ID_COL, CITY_COL]],
    on=HPG_RESTAURANT_ID_COL
)
hpg_res_sum = add_neighbors_reserves(hpg_res_sum, RESERVE_HPG, CITY_COL, RESERVE_HPG_NEIGHBORS)
hpg_res_sum.head()

,hpg_store_id,visit_date,hpg_reserves_sum,city,hpg_reserves_sum_neighbors
0,hpg_001ce40a1f873e4f,2016-01-13,4,Hyōgo-ken,5.914286
1,hpg_001ce40a1f873e4f,2016-01-27,7,Hyōgo-ken,6.173913
2,hpg_001ce40a1f873e4f,2016-02-13,2,Hyōgo-ken,5.962617
3,hpg_001ce40a1f873e4f,2016-02-27,8,Hyōgo-ken,8.017699
4,hpg_001ce40a1f873e4f,2016-03-16,2,Hyōgo-ken,6.568966


In [10]:
hpg_res_sum[RESERVE_HPG_NEIGHBORS].isna().sum()

np.int64(173)

In [11]:
hpg_res_sum_mapped = hpg_res_sum.merge(
    store_rel_df,
    on=HPG_RESTAURANT_ID_COL
)

### Date info

It is necessary to add a feature for the distance to the nearest holiday.

In [12]:
date_info_df = add_holiday_columns(date_info_df, CALENDAR_DATE_COL)

It is also necessary to designate Golden Week, since not all days of this week are holidays.

In [13]:
date_info_df = add_golden_week_flg(date_info_df, [2016, 2017], CALENDAR_DATE_COL)

### Air visit

#### Opening dates

We can consider the restaurant's opening date. Since simply counting the number of days since opening may not be sufficient due to the increase in days over time, it's best to flag the restaurant's opening as occurring within the last six months. A **potential issue**: some restaurants either planned to open earlier than their minimum opening date but didn't, or didn't report visitors for earlier dates. This is indicated by the fact that air_reserve dataframe has reservations for earlier dates.

In [14]:
air_open_dates = air_visit_df.groupby(AIR_RESTAURANT_ID_COL)[VISIT_DATE_COL].min().rename(OPEN_DATE_COL)

In [15]:
air_visit_df = add_opened_recently_flg(air_visit_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL)
sample_submission_df = add_opened_recently_flg(sample_submission_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL)

In [16]:
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_within_last_six_months_flg
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1


In [17]:
sample_submission_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_within_last_six_months_flg
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0


#### Days since the last visit record

Let's say if a restaurant was open on the previous day and the current day, the value will be zero. Otherwise, it will be the number of days since the last opening plus one. An open day is defined as a day when the restaurant has more than zero customers (in the original dataset, there are always more than zero customers).

However, sample_submission has a problem: it doesn't have a concept of gaps in dates, meaning it's impossible to definitively determine when restaurants were open or closed. We must either rely on further calculation of the opening hours or remove this feature altogether.

In [18]:
air_visit_df = add_open_flg(air_visit_df, VISITORS_COL)

In [19]:
air_visit_df = add_days_since_last_record(air_visit_df, AIR_RESTAURANT_ID_COL, VISIT_DATE_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_within_last_six_months_flg,is_open_flg,days_since_last_visit_record
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,1,0
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,1,0
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,0,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,1,2
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,1,0


#### Mean visitors by air city/region.

It is necessary to make smoothed target encoding, while the average value will be used for new areas. In this case, it is worth considering only days when the number of visitors is not zero, so that only open restaurants are taken into account.

In [20]:
air_visit_df = air_visit_df.merge(
    air_store_df,
    on=AIR_RESTAURANT_ID_COL
)

In [21]:
sample_submission_df = sample_submission_df.merge(
    air_store_df,
    on=AIR_RESTAURANT_ID_COL
)

In [22]:
air_visit_df, sample_submission_df = add_time_based_target_encoding(
    air_visit_df,
    sample_submission_df,
    CITY_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    AREA_TE
)

In [23]:
air_visit_df[[AIR_RESTAURANT_ID_COL, VISITORS_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]].head()

,air_store_id,visitors,air_city_region_te,city,visit_date
0,air_00a91d42b08b08d9,35,21.325809,Tōkyō-to,2016-07-01
1,air_00a91d42b08b08d9,9,21.390862,Tōkyō-to,2016-07-02
2,air_00a91d42b08b08d9,0,21.449328,Tōkyō-to,2016-07-03
3,air_00a91d42b08b08d9,20,21.455206,Tōkyō-to,2016-07-04
4,air_00a91d42b08b08d9,25,21.393245,Tōkyō-to,2016-07-05


In [24]:
sample_submission_df[[AIR_RESTAURANT_ID_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]].head()

,air_store_id,air_city_region_te,city,visit_date
0,air_00a91d42b08b08d9,20.47224,Tōkyō-to,2017-04-23
1,air_00a91d42b08b08d9,20.47224,Tōkyō-to,2017-04-24
2,air_00a91d42b08b08d9,20.47224,Tōkyō-to,2017-04-25
3,air_00a91d42b08b08d9,20.47224,Tōkyō-to,2017-04-26
4,air_00a91d42b08b08d9,20.47224,Tōkyō-to,2017-04-27


#### Mean visitors by air genre

It is necessary to make smoothed target encoding, while the average value will be used for new genres.

In [25]:
air_visit_df, sample_submission_df = add_time_based_target_encoding(
    air_visit_df,
    sample_submission_df,
    AIR_GENRE_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    GENRE_TE
)

In [26]:
air_visit_df[[AIR_RESTAURANT_ID_COL, VISITORS_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,visitors,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,35,22.490114,Italian/French,2016-07-01
1,air_00a91d42b08b08d9,9,22.540151,Italian/French,2016-07-02
2,air_00a91d42b08b08d9,0,22.600815,Italian/French,2016-07-03
3,air_00a91d42b08b08d9,20,22.619089,Italian/French,2016-07-04
4,air_00a91d42b08b08d9,25,22.563322,Italian/French,2016-07-05


In [27]:
sample_submission_df[[AIR_RESTAURANT_ID_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-23
1,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-24
2,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-25
3,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-26
4,air_00a91d42b08b08d9,22.565297,Italian/French,2017-04-27


#### Total reserved visitors


Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

By neighbors, we designate restaurants that are located in the same city.

In [28]:
air_visit_df = add_total_reserves(air_visit_df, air_res_sum, hpg_res_sum_mapped, CITY_COL)
air_visit_df = add_total_neigh_reserves(air_visit_df, hpg_res_sum, CITY_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_within_last_six_months_flg,is_open_flg,days_since_last_visit_record,air_genre_name,air_area_name,latitude,longitude,city,air_city_region_te,air_genre_te,total_reserves_sum,total_reserves_sum_neighbors
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,1,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,21.325809,22.490114,0.0,8.364261
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,1,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,21.390862,22.540151,0.0,7.995736
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,0,1,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,21.449328,22.600815,0.0,6.633588
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,1,2,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,21.455206,22.619089,0.0,6.715054
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,1,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,21.393245,22.563322,0.0,6.875000


In [29]:
sample_submission = add_total_reserves(sample_submission_df, air_res_sum, hpg_res_sum_mapped, CITY_COL)
sample_submission = add_total_neigh_reserves(sample_submission, hpg_res_sum, CITY_COL)
sample_submission.head()

,id,visitors,air_store_id,visit_date,open_date,opened_within_last_six_months_flg,air_genre_name,air_area_name,latitude,longitude,city,air_city_region_te,air_genre_te,total_reserves_sum,total_reserves_sum_neighbors
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,20.47224,22.565297,0.0,7.456000
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,20.47224,22.565297,0.0,10.395288
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,20.47224,22.565297,0.0,8.468889
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,20.47224,22.565297,0.0,11.811828
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0,Italian/French,Tōkyō-to Chiyoda-ku Kudanminami,35.694003,139.753595,Tōkyō-to,20.47224,22.565297,0.0,9.415233


### Rolling mean/median/std of visitors

Rolling values are NaN until the window is full. If EXCLUDE_ZEROS in config.py is True, zeros are treated as NaN and ignored in statistics. If the window contains only NaNs, the result is 0.

In [30]:
air_visit_df = add_basic_stats(air_visit_df, VISITORS_COL, AIR_RESTAURANT_ID_COL)
air_visit_df = add_neighbors_stats(air_visit_df, VISITORS_COL, CITY_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_within_last_six_months_flg,is_open_flg,days_since_last_visit_record,air_genre_name,air_area_name,latitude,...,visitors_neighbors,visitors_neighbors_mean_7,visitors_neighbors_median_7,visitors_neighbors_std_7,visitors_neighbors_mean_14,visitors_neighbors_median_14,visitors_neighbors_std_14,visitors_neighbors_mean_28,visitors_neighbors_median_28,visitors_neighbors_std_28
0,air_c31472d14e29cee8,2016-01-01,3,2016-01-01,1,1,0,Cafe/Sweets,Fukuoka-ken Fukuoka-shi Daimyō,33.589216,...,22.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,air_db80363d35f10926,2016-01-01,8,2016-01-01,1,1,0,Dining bar,Hokkaidō Asahikawa-shi 6 Jōdōri,43.770635,...,6.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,air_f690c42545146e0a,2016-01-01,7,2016-01-01,1,1,0,Japanese food,Hokkaidō Sapporo-shi Minami 3 Jōnishi,43.056819,...,6.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,air_efc80d3f96b3aff7,2016-01-01,10,2016-01-01,1,1,0,Other,Tōkyō-to Suginami-ku Asagayaminami,35.699566,...,24.400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,air_db4b38ebe7a7ceff,2016-01-01,21,2016-01-01,1,1,0,Dining bar,Ōsaka-fu Ōsaka-shi Shinmachi,34.676231,...,19.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Restaurant's opening days

A one-hot encoding for the days of the week is needed. This data can be extracted based on the percentage of gaps (after processing, 0 visitors per day). In this case, there should most likely be no reservations for that day.

## Conclusion

### Added features.

- Days to/from the nearest holiday (negative is days until, positive is days after).
- Holiday indicator (currently included in date_info).
- Separate indicators for Golden Week dates.

These features are currently in date_info dataframe, but will later be merged with air_visit.

- Indicator of whether the restaurant has been open within the last 6 months.
- Days since last recorded visit for air_visit dataframe.

- Rolling mean/median/std of visitors over the past week, month - for this restaurant/for neighbors.

- Smoothed target encoding of visitors by air_area_name.
- Smoothed target encoding of visitors by air_genre_name.

- Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

### Features planned for addition.

- Operating schedule - regular working days, extracted from the regular gaps in the data frame.

- Number of visitors on this day last month for this restaurant.
- Lag 1, 7, 28 of the number of visitors - for this restaurant/for neighbors.
- Missing lag indicator.

- Days since last recorded visit for sample_submission.

- Historical day-of-week mean/median up to (but not including) the current day - for this restaurant/for neighbors.

- Rolling reserve/visitors difference over the past 7 / 28 days - for this restaurant/for neighbors.

### Possible features.

- Daily temperature - try to get the weather forecast.
- Precipitation probability.
- Hpg genre.

Decomposition features:
- Trend.
- Trend difference for last month.
- Seasonal.
- Residual mean for last month.